# Create AI/BI Genie space (semantic NL layer)

Post-gold, post-metric-views step (idempotent **find-or-create by title**). Creates/updates an
**AI/BI Genie space** over the gold star schema so business users can ask questions in natural
language. It curates the space to the facts, conformed dimensions, and the two UC metric views.

**Optional:** the space needs a backing SQL warehouse (`warehouse_id`). If none is supplied this
notebook **exits cleanly without failing the job** — the rest of the sample is unaffected.

---

**Why a notebook (via the Genie API) rather than declaring it in the bundle?**

Genie space support is **new and still evolving**, and this approach keeps the sample runnable for
everyone: not all users are on the latest Databricks CLI, and some cannot upgrade it in their
environment. A notebook task is universally supported, so the sample deploys on any CLI version. We
therefore use this approach for now.

**Forward-looking:** the `serialized_space` config below is authored as a self-contained config
block, so as native bundle support for Genie spaces matures it can move into a
`resources/.../genie/*.yml` file and this notebook task be retired.

In [ ]:
# Genie space management APIs (create/list/update/trash_space) are relatively new; ensure a
# recent SDK regardless of the runtime's bundled version.
%pip install --quiet --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema_namespace", "tpch_sample")
dbutils.widgets.text("logical_env", "")
dbutils.widgets.text("warehouse_id", "")

catalog = dbutils.widgets.get("catalog")
schema_namespace = dbutils.widgets.get("schema_namespace")
logical_env = dbutils.widgets.get("logical_env")
warehouse_id = dbutils.widgets.get("warehouse_id").strip()

# Genie is OPTIONAL — no warehouse means skip cleanly (do NOT fail the job).
if not warehouse_id:
    print("No warehouse_id provided — skipping Genie space deployment.")
    dbutils.notebook.exit("skipped: no warehouse_id")

In [ ]:
import hashlib
import json

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

gold_schema = f"{schema_namespace}_gold{logical_env}"
g = f"{catalog}.{gold_schema}"
title = f"TPC-H Sample - Gold Analytics{logical_env}"
description = "Natural-language analytics over the TPC-H gold star schema (dimensions, facts, metric views)."


def _id(seed):
    # 32-char lowercase hex (UUID-without-hyphens format). Deterministic so re-deploys are
    # idempotent and the sorted-by-id collections below stay stable across runs.
    return hashlib.md5(seed.encode()).hexdigest()


def fq(table):
    return f"{g}.{table}"


# --- data sources (must be pre-sorted by identifier) ---
tables = sorted(
    [
        {"identifier": fq("fct_order_lines"),
         "description": ["Order-line grain sales fact. Measures: extended_price (gross), net_sales (net of discount), quantity."]},
        {"identifier": fq("fct_part_supply"),
         "description": ["Supplier inventory fact (part x supplier). Inventory value = available_qty * supply_cost."]},
        {"identifier": fq("dim_customer"),
         "description": ["Customer dimension (SCD2). Attributes: market_segment, nation_name, region_name."]},
        {"identifier": fq("dim_supplier"),
         "description": ["Supplier dimension. Attributes: supplier_name, nation_name, region_name."]},
        {"identifier": fq("dim_part"),
         "description": ["Part dimension. Attributes: brand, part_type, manufacturer."]},
        {"identifier": fq("dim_date"),
         "description": ["Date dimension. Attributes: year, quarter, month."]},
    ],
    key=lambda t: t["identifier"],
)

metric_views = sorted(
    [
        {"identifier": fq("tpch_sample_sales_metrics"),
         "description": ["Governed sales semantics. Query measures via MEASURE(...), e.g. Net Sales, On-Time Delivery Rate."]},
        {"identifier": fq("tpch_sample_supply_metrics"),
         "description": ["Governed supply semantics. Query measures via MEASURE(...), e.g. Inventory Value, Available Units."]},
    ],
    key=lambda t: t["identifier"],
)

# --- single high-level text instruction (at most 1 allowed) ---
instruction_text = (
    "This space analyzes the TPC-H gold star schema. Facts: fct_order_lines (order-line sales; "
    "net_sales is net of discount, extended_price is gross) and fct_part_supply (supplier "
    "inventory; inventory value = available_qty * supply_cost). Dimensions: dim_customer (SCD2), "
    "dim_supplier, dim_part, dim_date. Facts join to dimensions on surrogate keys (customer_sk, "
    "supplier_sk, part_sk) and to dim_date on order_date_key = date_key; these are resolved "
    "point-in-time, so dimension attributes reflect the version effective as of the order date. "
    "Prefer the metric views (tpch_sample_sales_metrics, tpch_sample_supply_metrics) for standard measures."
)
text_instructions = [{"id": _id("instruction:main"), "content": [instruction_text]}]


# --- example question -> SQL (must be pre-sorted by id) ---
def q_sql(question, sql_lines):
    return {"id": _id("eq:" + question), "question": [question], "sql": sql_lines}


example_question_sqls = sorted(
    [
        # Raw star-schema query (explicit joins + SUM) — shows the underlying model.
        q_sql(
            "What were net sales by market segment in 1996? (star schema)",
            [
                "SELECT c.market_segment, SUM(f.net_sales) AS net_sales\n",
                f"FROM {g}.fct_order_lines f\n",
                f"JOIN {g}.dim_customer c ON f.customer_sk = c.customer_sk\n",
                f"JOIN {g}.dim_date d ON f.order_date_key = d.date_key\n",
                "WHERE d.year = 1996\n",
                "GROUP BY c.market_segment\n",
                "ORDER BY net_sales DESC",
            ],
        ),
        # Same question via the governed sales metric view (MEASURE over pre-defined semantics).
        q_sql(
            "Net sales and on-time delivery rate by market segment in 1996 (governed metrics)",
            [
                "SELECT `Market Segment`,\n",
                "       MEASURE(`Net Sales`) AS net_sales,\n",
                "       MEASURE(`On-Time Delivery Rate`) AS on_time_delivery_rate\n",
                f"FROM {g}.tpch_sample_sales_metrics\n",
                "WHERE `Order Year` = 1996\n",
                "GROUP BY `Market Segment`\n",
                "ORDER BY net_sales DESC",
            ],
        ),
        # Raw star-schema query for supplier inventory.
        q_sql(
            "Top 10 suppliers by inventory value (star schema)",
            [
                "SELECT s.supplier_name, SUM(ps.available_qty * ps.supply_cost) AS inventory_value\n",
                f"FROM {g}.fct_part_supply ps\n",
                f"JOIN {g}.dim_supplier s ON ps.supplier_sk = s.supplier_sk\n",
                "GROUP BY s.supplier_name\n",
                "ORDER BY inventory_value DESC\n",
                "LIMIT 10",
            ],
        ),
        # Same question via the governed supply metric view.
        q_sql(
            "Top 10 suppliers by inventory value (governed metrics)",
            [
                "SELECT `Supplier`, MEASURE(`Inventory Value`) AS inventory_value\n",
                f"FROM {g}.tpch_sample_supply_metrics\n",
                "GROUP BY `Supplier`\n",
                "ORDER BY inventory_value DESC\n",
                "LIMIT 10",
            ],
        ),
    ],
    key=lambda e: e["id"],
)

# --- sample questions (must be pre-sorted by id) ---
sample_questions = sorted(
    [{"id": _id("sq:" + q), "question": [q]} for q in [
        "What were net sales by market segment last year?",
        "Top 10 suppliers by inventory value",
        "On-time delivery rate by ship mode",
        "Which brands have the highest return rate?",
    ]],
    key=lambda e: e["id"],
)

space = {
    "version": 2,
    "config": {"sample_questions": sample_questions},
    "data_sources": {"tables": tables, "metric_views": metric_views},
    "instructions": {
        "text_instructions": text_instructions,
        "example_question_sqls": example_question_sqls,
    },
}
serialized = json.dumps(space)


def find_space_id(space_title):
    """Find an existing space by exact title (paginated) so re-runs update in place."""
    token = None
    while True:
        resp = w.genie.list_spaces(page_size=100, page_token=token)
        for s in (resp.spaces or []):
            if s.title == space_title:
                return s.space_id
        token = getattr(resp, "next_page_token", None)
        if not token:
            return None


space_id = find_space_id(title)
if space_id:
    # etag omitted -> unconditional update (full serialized_space replacement).
    w.genie.update_space(space_id=space_id, serialized_space=serialized, title=title, warehouse_id=warehouse_id)
    print(f"Updated existing Genie space: {space_id}")
else:
    try:
        parent_path = f"/Workspace/Users/{w.current_user.me().user_name}"
    except Exception:
        parent_path = None
    created = w.genie.create_space(
        warehouse_id=warehouse_id, serialized_space=serialized,
        title=title, description=description, parent_path=parent_path,
    )
    space_id = created.space_id
    print(f"Created Genie space: {space_id}")

print(f"Genie space ready: {w.config.host}/genie/rooms/{space_id}")